[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/huggingface-nlp-certified/notebooks/day-06-evaluate-metrics.ipynb#scrollTo=c1d2e3f4)

---
# Day 6 · The Evaluate Library — Metrics After Fine-Tuning (F1, BLEU, ROUGE)
**certified-journeys / huggingface-nlp-certified** · Day 6 · Evaluation

> **Goal for today:** By the end of this notebook you can load and compute accuracy, F1, ROUGE, and BLEU using the `evaluate` library, understand what each metric measures, and use `evaluator.compute()` to evaluate a pipeline end-to-end against a dataset split.


In [ ]:
%pip install -q evaluate transformers datasets torch sacrebleu rouge_score


## Step 1 · The Evaluate library — what it is and how it works

The `evaluate` library provides a standardised, reproducible interface to 100+ evaluation metrics. Instead of writing metric logic yourself, you:
1. `evaluate.load(metric_name)` — download and cache the metric
2. Call `.compute(predictions=..., references=...)` — get a result dict

**Reference:** [Evaluate library quickstart](https://huggingface.co/docs/evaluate/index)

| Metric | Task | Input format |
|---|---|---|
| `accuracy` | Classification | Integer predictions vs integer references |
| `f1` | Classification | Integer predictions vs integer references |
| `rouge` | Summarization | String predictions vs string references |
| `sacrebleu` | Translation | String predictions vs list-of-string references |
| `bertscore` | Semantic similarity | String predictions vs string references |


In [ ]:
import evaluate

# Load the accuracy metric
accuracy_metric = evaluate.load("accuracy")

# Compute accuracy: 2/3 correct → 0.667
result = accuracy_metric.compute(
    predictions=[1, 0, 1],
    references=[1, 1, 1],
)

print("Accuracy result:", result)
print("Accuracy value :", result["accuracy"])


In [ ]:
# Inspect metric metadata
print("Metric description (first 200 chars):\n", accuracy_metric.description[:200])
print()
# All metrics expose features describing the required input format
print("Input features:", accuracy_metric.features)


### What just happened?
- `evaluate.load('accuracy')` downloads a tiny Python module from the Hub and caches it — the same caching mechanism as `load_dataset`.
- `.compute()` is stateless — it takes predictions and references in a single call. Some metrics also support incremental `.add_batch()` for streaming evaluation.
- Every metric returns a **dict** — even if there's only one value. Always access `result["accuracy"]` not `result` directly.
- `.description` and `.features` are available on every metric — useful for understanding expected input shapes before wiring up your evaluation loop.


## Step 2 · F1 score for multi-class classification

Accuracy is misleading on imbalanced datasets — a model predicting the majority class 100% of the time can score high. **F1** balances precision and recall.

| Average mode | What it computes |
|---|---|
| `macro` | Unweighted mean of per-class F1 — treats all classes equally |
| `micro` | Global TP/FP/FN counts — dominated by frequent classes |
| `weighted` | Mean weighted by class support — accounts for imbalance |
| `binary` | F1 for the positive class only (default for binary tasks) |

For multi-class evaluation with class imbalance, **macro-F1** is the standard choice because it forces the model to perform well across all classes.


In [ ]:
f1_metric = evaluate.load("f1")

# Simulate 3-class classification (e.g., negative / neutral / positive)
preds_3class    = [0, 1, 2, 0, 1, 2, 1, 0, 2, 1]
refs_3class     = [0, 0, 2, 0, 1, 1, 1, 2, 2, 1]

# Macro-averaged F1 — treats all 3 classes equally regardless of size
result_macro = f1_metric.compute(
    predictions=preds_3class,
    references=refs_3class,
    average="macro",
)
print("Macro F1:", result_macro)

# Weighted F1 — accounts for class imbalance
result_weighted = f1_metric.compute(
    predictions=preds_3class,
    references=refs_3class,
    average="weighted",
)
print("Weighted F1:", result_weighted)


In [ ]:
# Per-class F1 breakdown using sklearn (complementary to evaluate)
from sklearn.metrics import classification_report

print(classification_report(
    refs_3class,
    preds_3class,
    target_names=["negative", "neutral", "positive"],
))


### What just happened?
- **Macro F1** is the unweighted mean of per-class F1 — if class 2 scores 0.0, it pulls the macro average down even if classes 0 and 1 are perfect. This is intentional: it penalises models that ignore minority classes.
- **Weighted F1** weights each class's F1 by its proportion in the reference set — it's a fairer single number when class imbalance is extreme but expected in production.
- `classification_report` (sklearn) gives the per-class breakdown that `evaluate.load('f1')` doesn't provide — use both together for a complete picture.
- The `evaluate` metric and sklearn agree on the value — always cross-check a new metric with a known implementation.


## Step 3 · ROUGE for summarization

**ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) measures n-gram overlap between model output and reference summaries.

| Variant | What it measures |
|---|---|
| `rouge1` | Unigram (single word) overlap |
| `rouge2` | Bigram overlap — captures phrase-level fluency |
| `rougeL` | Longest common subsequence — rewards order |

Each variant returns `precision`, `recall`, and **`fmeasure`** (F1). The standard reported number is `fmeasure`.

**Reference:** [Choosing a metric](https://huggingface.co/docs/evaluate/choosing_a_metric)


In [ ]:
rouge_metric = evaluate.load("rouge")

# Simulate model-generated summaries vs human reference summaries
predictions = [
    "The transformer model achieved state-of-the-art results on multiple benchmarks.",
    "Scientists discovered a new method for protein folding prediction.",
    "The company reported record quarterly profits in its earnings call.",
]
references = [
    "A new transformer architecture set state-of-the-art performance across several NLP tasks.",
    "Researchers developed an AI-based protein structure prediction algorithm.",
    "The firm announced its highest-ever quarterly revenue during the investor meeting.",
]

# rouge_types controls which variants to compute
# stemming=True normalises word forms (running → run) for consistent comparison
result = rouge_metric.compute(
    predictions=predictions,
    references=references,
    rouge_types=["rouge1", "rouge2", "rougeL"],
    use_stemmer=True,
)

print("ROUGE scores (aggregated fmeasure across examples):")
for key, value in result.items():
    print(f"  {key:8s}: {value:.4f}")


In [ ]:
# Inspect per-example scores using the raw rouge_score library directly
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

print("Per-example ROUGE scores:")
for i, (pred, ref) in enumerate(zip(predictions, references)):
    scores = scorer.score(ref, pred)
    r1 = scores["rouge1"].fmeasure
    r2 = scores["rouge2"].fmeasure
    rL = scores["rougeL"].fmeasure
    print(f"  Example {i+1}: ROUGE-1={r1:.3f}  ROUGE-2={r2:.3f}  ROUGE-L={rL:.3f}")


### What just happened?
- `evaluate.load('rouge')` uses the `rouge_score` package under the hood — the same library used in most summarization papers.
- **`use_stemmer=True`** strips word suffixes before comparing ("achieved" → "achiev") — without it, "achieves" and "achieved" count as different words, making scores non-comparable across runs with different capitalization or tense.
- **ROUGE-1** tells you if the key words are present; **ROUGE-2** tells you if key phrases are in the right order; **ROUGE-L** is more lenient about word order than ROUGE-2.
- The `evaluate` library aggregates by taking the **mean** across examples by default — always check whether your task needs per-example or corpus-level aggregation.


## Step 4 · BLEU for machine translation with sacrebleu

**BLEU** (Bilingual Evaluation Understudy) measures precision of n-grams in the hypothesis against one or more reference translations. It ranges 0–100 (higher is better).

**SacreBLEU** is the recommended implementation — it standardises tokenization so scores are comparable across papers and systems.

Key differences from ROUGE:
- BLEU is **precision**-oriented; ROUGE is **recall**-oriented
- BLEU accepts **multiple** reference translations per source sentence
- BLEU has a **brevity penalty** that penalises short hypotheses


In [ ]:
bleu_metric = evaluate.load("sacrebleu")

# Three MT system outputs (hypotheses)
mt_predictions = [
    "The cat sat on the mat.",
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning models require large amounts of training data.",
]

# References: each prediction can have MULTIPLE reference translations
# sacrebleu expects: list of lists — references[i] is a list of refs for prediction[i]
mt_references = [
    ["The cat is sitting on the mat.", "A cat sat upon the mat."],
    ["The fast brown fox leaps over the lazy dog."],
    ["Training machine learning models needs large datasets.", "ML models need lots of training data."],
]

result = bleu_metric.compute(
    predictions=mt_predictions,
    references=mt_references,
)

print("SacreBLEU result:")
print(f"  BLEU score     : {result['score']:.2f}")
print(f"  Brevity penalty: {result['bp']:.4f}")
print(f"  Length ratio   : {result['ratio']:.4f}")
print(f"  N-gram precisions (1-4): {[round(p, 2) for p in result['precisions']]}")


In [ ]:
# Demonstrate how reference count affects BLEU
# One reference: harder to match → lower score
result_one_ref = bleu_metric.compute(
    predictions=mt_predictions,
    references=[[r[0]] for r in mt_references],  # keep only first reference
)

# Multiple references: more ways to match → higher score
print(f"BLEU with 1 reference per example   : {result_one_ref['score']:.2f}")
print(f"BLEU with multiple references        : {result['score']:.2f}")
print("Multiple references always improve BLEU — use all available when evaluating.")


### What just happened?
- **SacreBLEU** is the standard for reproducible BLEU evaluation — it tokenises consistently regardless of whether your translations already have spaces around punctuation.
- References in sacrebleu are passed as `[[ref1_a, ref1_b], [ref2_a], ...]` — a list of lists where the outer list aligns with predictions.
- **Brevity penalty (bp)** is 1.0 when the hypothesis is at least as long as the reference; it exponentially penalises shorter outputs — this prevents the model from gaming BLEU by generating just a few high-precision words.
- N-gram precisions: `precisions[0]` is unigram, `precisions[3]` is 4-gram — BLEU is the geometric mean of all four, so poor 4-gram precision tanks the overall score even if unigrams look good.


## Step 5 · End-to-end evaluation with `evaluator.compute()`

For common task types, `evaluate.evaluator()` provides an end-to-end evaluator that:
1. Wraps a `pipeline`
2. Iterates over a dataset split
3. Computes the metric automatically

This removes the boilerplate of writing an evaluation loop manually.

**Reference:** [Choosing a metric](https://huggingface.co/docs/evaluate/choosing_a_metric)

| `task` string | Pipeline type | Required columns |
|---|---|---|
| `"text-classification"` | TextClassificationPipeline | `text`, `label` |
| `"token-classification"` | TokenClassificationPipeline | `tokens`, `ner_tags` |
| `"question-answering"` | QuestionAnsweringPipeline | `question`, `context`, `answers` |


In [ ]:
from transformers import pipeline
from datasets import load_dataset
import evaluate

# Load a fine-tuned sentiment classifier as a pipeline
clf_pipe = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=-1,  # -1 = CPU; use 0 for GPU if available
)

# Use a small slice of SST-2 from the GLUE benchmark
sst2 = load_dataset("glue", "sst2", split="validation[:50]")
print("Dataset columns:", sst2.column_names)
print("Num examples   :", len(sst2))
print("Label names    :", sst2.features["label"].names)


In [ ]:
# The SST-2 dataset uses 'sentence' not 'text', and labels are 0/1
# The pipeline returns dicts like {'label': 'POSITIVE', 'score': 0.999}
# evaluator needs to know how to map pipeline output to integer labels

task_evaluator = evaluate.evaluator("text-classification")

eval_results = task_evaluator.compute(
    model_or_pipeline=clf_pipe,
    data=sst2,
    metric=evaluate.load("accuracy"),
    label_mapping={"NEGATIVE": 0, "POSITIVE": 1},  # map pipeline str labels → int
    input_column="sentence",   # column name for the input text
    label_column="label",      # column name for the reference labels
)

print("End-to-end evaluator results:")
for key, value in eval_results.items():
    print(f"  {key}: {value}")


In [ ]:
# Compare with manual evaluation loop to verify
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

manual_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
manual_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased-finetuned-sst-2-english"
)

all_preds = []
all_refs  = []

for example in sst2:
    inputs = manual_tokenizer(example["sentence"], return_tensors="pt", truncation=True)
    with torch.no_grad():
        logits = manual_model(**inputs).logits
    pred = logits.argmax(dim=-1).item()
    all_preds.append(pred)
    all_refs.append(example["label"])

manual_accuracy = evaluate.load("accuracy").compute(
    predictions=all_preds,
    references=all_refs,
)
print("Manual loop accuracy :", manual_accuracy)
print("Evaluator accuracy   :", {"accuracy": eval_results["accuracy"]})
print("Match:", abs(manual_accuracy["accuracy"] - eval_results["accuracy"]) < 1e-6)


### What just happened?
- **`evaluate.evaluator`** is a high-level convenience wrapper that handles batch inference, output format conversion, and metric computation in one call — it replaces a 20-line evaluation loop.
- **`label_mapping`** bridges the gap between pipeline string outputs (`"POSITIVE"`) and integer reference labels — always check this mapping when evaluation results look surprisingly wrong.
- The manual loop and the evaluator produce **identical accuracy** — the evaluator is just a tested abstraction over the same operations.
- For production evaluation, prefer the manual loop when you need fine-grained control (e.g., logging per-example scores, handling exceptions, custom batching); use `evaluator.compute()` for quick benchmark comparisons.


In [ ]:
# Challenge: Evaluate a summarization pipeline end-to-end using ROUGE.
# Use the 'sshleifer/distilbart-cnn-6-6' model on the first 10 examples
# of the 'xsum' dataset. Compute rouge1, rouge2, rougeL with use_stemmer=True.
# Print the three ROUGE fmeasure scores.
#
# Hints:
#   - xsum has columns: 'document' (article text), 'summary' (reference)
#   - pipeline task = "summarization"
#   - Use max_length=60, min_length=10 for the pipeline to keep it fast
#   - Collect pipeline outputs into a predictions list of strings
#   - Collect dataset 'summary' column into a references list of strings
#   - Then call rouge_metric.compute(predictions=..., references=...,
#       rouge_types=[...], use_stemmer=True)
#
# Scaffold:
# from transformers import pipeline
# from datasets import load_dataset
# import evaluate
#
# summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-6-6", device=-1)
# xsum = load_dataset("xsum", split="test[:10]")
#
# predictions = []
# references  = []
# for example in xsum:
#     out = summarizer(example["document"], max_length=60, min_length=10)
#     predictions.append(...)  # TODO: extract generated text
#     references.append(...)
#
# rouge = evaluate.load("rouge")
# result = rouge.compute(...)  # TODO: fill in
# print(result)

# Your solution here


---
## Day 6 key concepts recap

| Concept | What to remember |
|---|---|
| `evaluate.load(name)` | Downloads & caches metric; all metrics return dicts |
| `.compute(predictions, references)` | Stateless call — pass all data at once |
| Accuracy | Right for balanced datasets; misleading on skewed label distributions |
| Macro F1 | Preferred multi-class metric — penalises models that ignore minority classes |
| ROUGE | n-gram recall for summarization; always use `use_stemmer=True` and specify `rouge_types` |
| SacreBLEU | Standard reproducible BLEU for MT; references are list-of-lists; multiple refs improve score |
| `evaluator.compute()` | End-to-end: pipeline + dataset + metric in one call; use `label_mapping` for string labels |

> **Tip:** ROUGE scores are notoriously sensitive to whitespace and case — always use `rouge_types=['rouge1','rouge2','rougeL']` and `use_stemmer=True` for consistent comparisons across runs.

---
## What's next
**Day 7** → Fine-Tuning with the Trainer API — putting tokenized datasets and evaluation metrics together to fine-tune a model end-to-end using `Trainer` and `TrainingArguments`.

Mark Day 6 complete in your [tracker](../index.html).
